In [ ]:
               

import os

import copy

import random

import collections

import itertools

import numpy as np

import pandas as pd

import warnings

import joblib

from sklearn.model_selection import train_test_split,RandomizedSearchCV

import sklearn.metrics as metrics

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer

from sklearn.utils import resample


In [ ]:
                    

base_dir = 'circExor/DL_models/circRNA_DL_Model_tridivided_intra5fold_Output'

model_folders = {

    'CNN': 'circCNN',

    'GRU': 'circGRU',

    'MLP': 'circMLP'

}

csv_metrics = ['AUROC', 'AUPRC', 'MCC', 'Precision', 'Recall', 'F1_Score']

display_metrics = ['AUROC', 'AUPRC', 'MCC', 'Precision', 'Recall', 'F1']

original_models = []

original_mean_data = []

original_std_data = []

             

for folder, model_name in model_folders.items():

    csv_path = os.path.join(base_dir, folder, '5fold_cv_metrics.csv')

    df = pd.read_csv(csv_path)

    df_metrics = df[csv_metrics]

    

    original_mean_data.append(df_metrics.mean().values)

    original_std_data.append(df_metrics.std().values)

    original_models.append(model_name)

original_mean_data = np.array(original_mean_data)

original_std_data = np.array(original_std_data)

          

desired_order = ['circMLP', 'circCNN', 'circGRU']

            

sorted_indices = []

for model in desired_order:

    if model in original_models:

        sorted_indices.append(original_models.index(model))

           

models = [original_models[i] for i in sorted_indices]

data_mean = original_mean_data[sorted_indices]

data_std = original_std_data[sorted_indices]

               

cool_colors = [

    '#D4E6F1', '#C5D9E8', '#B6CCDF', '#A7BFD6', '#98B2CD',

    '#89A5C4', '#7A98BB', '#6B8BB2', '#5C7EA9', '#4D71A0',

    '#3E6497', '#2F578E', '#204A85', '#113D7C'

]

              

model_colors = [cool_colors[i] for i in [1, 5,  9]]

        

fig, ax = plt.subplots(figsize=(14, 8))

n_metrics = len(display_metrics)

n_models = len(models)

bar_width = 0.12

x = np.arange(n_metrics)

               

for i, model in enumerate(models):

    offset = (i - n_models/2 + 0.5) * bar_width

    bars = ax.bar(x + offset, data_mean[i], bar_width, 

                  yerr=data_std[i], capsize=3,

                  label=model, color=model_colors[i], 

                  edgecolor='black', linewidth=0.8,

                  error_kw=dict(lw=1, capthick=1, alpha=0.7))

        

ax.set_xticks(x)

ax.set_xticklabels(display_metrics, fontsize=16)

ax.tick_params(axis='y', labelsize=14)

ax.set_ylim(0, 1.0)

ax.grid(axis='y', alpha=0.3, linestyle='--')

ax.legend(fontsize=12, loc='upper left', bbox_to_anchor=(1, 1))

      

plt.tight_layout()

plt.show()

for i, model in enumerate(models):

    print(f"{i+1}. {model}: AUROC = {data_mean[i][0]:.4f} ± {data_std[i][0]:.4f}")


In [ ]:
        

                               

ml_base_dir = 'circExor/ML_models/circRNA_ML_Model_tridivided_intra5fold_Output'

ml_model_folders = {

    'LogisticRegression': 'LR',

    'SVM': 'SVM',

    'CatBoost_RandomSearch': 'CatBoost',

    'LightGBM': 'LightGBM',

    'RandomForest': 'Random_Forest',

    'NGBoost_RandomSearch': 'NGBoost'

}

dl_base_dir = 'circExor/DL_models/circRNA_DL_Model_tridivided_intra5fold_Output'

dl_model_folders = {

    'CNN': 'circCNN',

    'GRU': 'circGRU',

    'MLP': 'circMLP'

}

csv_metrics = ['AUROC', 'AUPRC', 'MCC', 'Precision', 'Recall', 'F1_Score']

display_metrics = ['AUROC', 'AUPRC', 'MCC', 'Precision', 'Recall', 'F1']

original_models = []

original_mean_data = []

original_std_data = []

                       

for folder, model_name in ml_model_folders.items():

    csv_path = os.path.join(ml_base_dir, folder, '5fold_cv_metrics.csv')

    if os.path.exists(csv_path):

        df = pd.read_csv(csv_path)

        df_metrics = df[csv_metrics]

        original_mean_data.append(df_metrics.mean().values)

        original_std_data.append(df_metrics.std().values)

        original_models.append(model_name)

    else:

        print(f"Warning: ML model data not found at {csv_path}")

                       

for folder, model_name in dl_model_folders.items():

    csv_path = os.path.join(dl_base_dir, folder, '5fold_cv_metrics.csv')

    if os.path.exists(csv_path):

        df = pd.read_csv(csv_path)

        df_metrics = df[csv_metrics]

        original_mean_data.append(df_metrics.mean().values)

        original_std_data.append(df_metrics.std().values)

        original_models.append(model_name)

    else:

        print(f"Warning: DL model data not found at {csv_path}")

original_mean_data = np.array(original_mean_data)

original_std_data = np.array(original_std_data)

                        

f1_idx = csv_metrics.index('F1_Score')

f1_values = original_mean_data[:, f1_idx]

                      

sorted_indices = np.argsort(f1_values)

           

models = [original_models[i] for i in sorted_indices]

data_mean = original_mean_data[sorted_indices]

data_std = original_std_data[sorted_indices]

               

cool_colors = [

    '#D4E6F1', '#C5D9E8', '#B6CCDF', '#A7BFD6', '#98B2CD',

    '#89A5C4', '#7A98BB', '#6B8BB2', '#5C7EA9', '#4D71A0',

    '#3E6497', '#2F578E', '#204A85', '#113D7C'

]

                    

color_indices = [0, 1, 3, 5, 7, 8, 10, 11, 13]

model_colors = [cool_colors[i] for i in color_indices]

                          

fig, ax = plt.subplots(figsize=(18, 8))

n_metrics = len(display_metrics)

n_models = len(models)

bar_width = 0.09              

x = np.arange(n_metrics)

               

for i, model in enumerate(models):

    offset = (i - n_models/2 + 0.5) * bar_width

    bars = ax.bar(x + offset, data_mean[i], bar_width, 

                  yerr=data_std[i], capsize=2,

                  label=model, color=model_colors[i], 

                  edgecolor='black', linewidth=0.8,

                  error_kw=dict(lw=1, capthick=1, alpha=0.7))

        

ax.set_xticks(x)

ax.set_xticklabels(display_metrics, fontsize=16)

ax.tick_params(axis='y', labelsize=14)

ax.set_ylim(0, 1.0)

ax.grid(axis='y', alpha=0.3, linestyle='--')

ax.legend(fontsize=12, loc='upper left', bbox_to_anchor=(1, 1))

      

plt.tight_layout()

plt.show()

                    

print("Models sorted by mean F1 from low to high:")

for i, model in enumerate(models):

    print(f"{i+1}. {model}: F1_Score = {data_mean[i][f1_idx]:.4f} ± {data_std[i][f1_idx]:.4f}")


In [ ]:
                   

models = [

    'circGRU', 'SVM', 'circCNN', 'LR', 'circMLP', 

    'NGBoost', 'LightGBM', 'CatBoost', 'Random_Forest'

]

                      

cool_colors = [

    '#D4E6F1', '#C5D9E8', '#B6CCDF', '#A7BFD6', '#98B2CD',

    '#89A5C4', '#7A98BB', '#6B8BB2', '#5C7EA9', '#4D71A0',

    '#3E6497', '#2F578E', '#204A85', '#113D7C'

]

             

color_indices = [0, 1, 3, 5, 7, 8, 10, 11, 13]

base_9_colors = [cool_colors[i] for i in color_indices]

               

light_to_dark_order = [

    'circGRU', 'SVM', 'circCNN', 'LR', 'circMLP', 

    'NGBoost', 'LightGBM', 'CatBoost', 'Random_Forest'

]

                             

model_colors = []

for model in models:

                          

    rank = light_to_dark_order.index(model)

               

    model_colors.append(base_9_colors[rank])

                    

def plot_model_legend(models, colors, filename, labelspacing=0.6, markersize=12, figsize=(3, 7)):                   

    """
    Draw the model-color legend and save it with a transparent background
    Designed for stacked bar plots
    """

    plt.rcParams['font.family'] = 'Arial'

    fig, ax = plt.subplots(figsize=figsize)

    handles = []

    for model, color in zip(models, colors):

                         

        patch = plt.Rectangle(

            (0, 0), 1, 1, facecolor=color, 

            edgecolor='black', linewidth=1,

            label=model

        )

        handles.append(patch)

    legend = ax.legend(

        handles=handles,

        labels=models,

        loc='center',

        ncol=1,

        frameon=False,

        labelspacing=labelspacing,         

        handlelength=1.5,                   

        handletextpad=0.8,                   

        fontsize=12                       

    )

    

                

    for handle in legend.legendHandles:

        handle.set_width(markersize/10)

        handle.set_height(markersize/15)

    

    ax.axis('off')

    plt.tight_layout()

    plt.savefig(filename, dpi=300, bbox_inches='tight', transparent=True)

    plt.show()

          

plot_model_legend(models, model_colors, "legend_models.png", labelspacing=0.8, markersize=200)


In [ ]:
                                                

from scipy import stats

from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import os

                                    

ml_base_dir = 'circExor/ML_models/circRNA_ML_Model_tridivided_intra5fold_Output'

ml_model_folders = {

    'LogisticRegression': 'LR',

    'SVM': 'SVM',

    'CatBoost_RandomSearch': 'CatBoost',

    'LightGBM': 'LightGBM',

    'RandomForest': 'Random_Forest',

    'NGBoost_RandomSearch': 'NGBoost'

}

dl_base_dir = 'circExor/DL_models/circRNA_DL_Model_tridivided_intra5fold_Output'

dl_model_folders = {

    'CNN': 'circCNN',

    'GRU': 'circGRU',

    'MLP': 'circMLP'

}

csv_metrics = ['AUROC', 'AUPRC', 'MCC', 'Precision', 'Recall', 'F1_Score']

display_metrics = ['AUROC', 'AUPRC', 'MCC', 'Precision', 'Recall', 'F1']

original_models = []

original_mean_data = []

original_std_data = []

original_raw_data = []                      

                        

            

for folder, model_name in ml_model_folders.items():

    csv_path = os.path.join(ml_base_dir, folder, '5fold_cv_metrics.csv')

    if os.path.exists(csv_path):

        df = pd.read_csv(csv_path)

        df_metrics = df[csv_metrics]

        original_raw_data.append(df_metrics.values) 

        original_mean_data.append(df_metrics.mean().values)

        original_std_data.append(df_metrics.std().values)

        original_models.append(model_name)

            

for folder, model_name in dl_model_folders.items():

    csv_path = os.path.join(dl_base_dir, folder, '5fold_cv_metrics.csv')

    if os.path.exists(csv_path):

        df = pd.read_csv(csv_path)

        df_metrics = df[csv_metrics]

        original_raw_data.append(df_metrics.values) 

        original_mean_data.append(df_metrics.mean().values)

        original_std_data.append(df_metrics.std().values)

        original_models.append(model_name)

original_mean_data = np.array(original_mean_data)

original_std_data = np.array(original_std_data)

original_raw_data = np.array(original_raw_data)                     

                                                        

n_folds = 5

df_t = n_folds - 1       

                                

t_critical = stats.t.ppf(1 - 0.05/2, df_t) 

original_ci_data = (original_std_data / np.sqrt(n_folds)) * t_critical

                             

f1_idx = csv_metrics.index('F1_Score')

f1_values = original_mean_data[:, f1_idx]

sorted_indices = np.argsort(f1_values)

           

models = [original_models[i] for i in sorted_indices]

data_mean = original_mean_data[sorted_indices]

data_ci = original_ci_data[sorted_indices]             

data_raw = original_raw_data[sorted_indices]

                    

               

cool_colors = [

    '#D4E6F1', '#C5D9E8', '#B6CCDF', '#A7BFD6', '#98B2CD',

    '#89A5C4', '#7A98BB', '#6B8BB2', '#5C7EA9', '#4D71A0',

    '#3E6497', '#2F578E', '#204A85', '#113D7C'

]

                    

color_indices = [0, 1, 3, 5, 7, 8, 10, 11, 13]

model_colors = [cool_colors[i] for i in color_indices]

                    

fig, ax = plt.subplots(figsize=(18, 9))

n_metrics = len(display_metrics)

n_models = len(models)

bar_width = 0.09                   

x = np.arange(n_metrics)

                                      

bar_x_positions = np.zeros((n_metrics, n_models))

               

for i, model in enumerate(models):

    offset = (i - n_models/2 + 0.5) * bar_width

    bar_x_positions[:, i] = x + offset

    bars = ax.bar(bar_x_positions[:, i], data_mean[i], bar_width, 

                  yerr=data_ci[i], capsize=2,                   

                  label=model, color=model_colors[i], 

                  edgecolor='black', linewidth=0.8,

                  error_kw=dict(lw=1, capthick=1, alpha=0.7))

                                      

def get_asterisks(p):

    if p < 0.001: return '***'

    elif p < 0.01: return '**'

    elif p < 0.05: return '*'

    return 'ns'

best_model_idx = n_models - 1                    

for m_idx in range(n_metrics):

    p_values = []

    compare_indices = []

    

                       

    best_raw = data_raw[best_model_idx, :, m_idx]

    

               

    for left_idx in range(n_models - 1):

        left_raw = data_raw[left_idx, :, m_idx]

                                         

        _, p_val = stats.ttest_rel(best_raw, left_raw, alternative='greater')

        p_values.append(p_val)

        compare_indices.append(left_idx)

        

                                     

    _, p_adj, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

    

               

                          

    base_y_offset = max(data_mean[:, m_idx] + data_ci[:, m_idx]) + 0.02

    for offset_idx, (left_idx, p_val) in enumerate(zip(compare_indices, p_adj)):

        stars = get_asterisks(p_val)

        if stars != 'ns':

            x1 = bar_x_positions[m_idx, left_idx]

            x2 = bar_x_positions[m_idx, best_model_idx]

                              

            y = base_y_offset + (n_models - 2 - offset_idx) * 0.03

            h = 0.01

            

                  

            ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, color='black')

                  

            ax.text((x1+x2)/2, y+h, stars, ha='center', va='bottom', color='black', fontsize=12)

                   

ax.set_xticks(x)

ax.set_xticklabels(display_metrics, fontsize=16)

ax.tick_params(axis='y', labelsize=14)

                     

ax.set_ylim(0, max(data_mean.flatten() + data_ci.flatten()) + (n_models * 0.04))

ax.grid(axis='y', alpha=0.3, linestyle='--')

ax.legend(fontsize=16, loc='upper right', ncol=5)

      

plt.tight_layout()

plt.show()

                                      

print("Models sorted by mean F1 from low to high (mean ± 95% CI):")

for i, model in enumerate(models):

    print(f"{i+1}. {model}: F1_Score = {data_mean[i][f1_idx]:.4f} ± {data_ci[i][f1_idx]:.4f}")


In [ ]:
import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import os

def plot_AUROC(main_path):

                                   

    plt.figure(figsize=(10, 10))              

    lw = 2

    

                    

    color_list = [

        (0.7, 0.4, 0.1),              

        (0.3, 0.6, 0.9),            

        (0.5, 0.7, 0.2),             

        (0.8, 0.5, 0.6),            

        (0.6, 0.3, 0.8),              

        (0.9, 0.7, 0.3),              

        (0.4, 0.8, 0.5),            

        (0.6, 0.4, 0.4),             

        (0.4, 0.4, 0.4)             

    ]

    

                                       

    model_paths = {}

    

              

    subfolders = sorted([d for d in os.listdir(main_path) if os.path.isdir(os.path.join(main_path, d))])

    for model in subfolders:

        model_paths[model] = os.path.join(main_path, model, 'AUROC_info.txt')

        

                         

    dl_base = 'circExor/DL_models/circRNA_DL_Model_tridivided_intra5fold_Output'

    model_paths['circCNN'] = os.path.join(dl_base, 'CNN', 'CNN_Final_Model', 'AUROC_info.txt')

    model_paths['circGRU'] = os.path.join(dl_base, 'GRU', 'GRU_Final_Model', 'AUROC_info.txt')

    model_paths['circMLP'] = os.path.join(dl_base, 'MLP', 'MLP_Final_Model', 'AUROC_info.txt')

    

             

    model_list = list(model_paths.keys())

    model_colors = {model: color_list[i % len(color_list)] for i, model in enumerate(model_list)}

    

                                                       

    def load_roc_data(file_path):

        df = pd.read_csv(file_path, sep='\t')

        fpr = df['FPR'].values

        tpr = df['TPR'].values

                                                              

        sorted_idx = np.argsort(fpr)

        fpr = fpr[sorted_idx]

        tpr = tpr[sorted_idx]

                                            

        roc_auc = np.trapz(tpr, fpr)

        return fpr, tpr, roc_auc

    

                                             

    def plot_single_roc(fpr, tpr, roc_auc, label, color):

        plt.plot(fpr, tpr, color=color, lw=lw, label=f'{label} (AUC = {roc_auc:.4f})')

    

                           

    for model, file_path in model_paths.items():

        if os.path.exists(file_path):

            fpr, tpr, roc_auc = load_roc_data(file_path)

            plot_single_roc(fpr, tpr, roc_auc, model, model_colors[model])

        else:

            print(f"Warning: File {file_path} not found. Skipping {model}.")

    

                                            

    plt.plot([0, 1], [0, 1], color='grey', lw=lw, linestyle='--')

                        

    plt.xlim([0.0, 1.0])

    plt.ylim([0.0, 1.05])

    plt.xticks(fontsize=17)

    plt.yticks(fontsize=17)

    plt.xlabel("False Positive Rate", fontsize=20)

    plt.ylabel("True Positive Rate", fontsize=20)

    plt.legend(loc="lower right", fontsize=14)

    

                   

    plt.savefig(os.path.join(main_path, "AUROC_All_Models.pdf"), format="pdf")

    plt.show()

                

plot_AUROC('circExor/circExor/models/saved_models/circRNA_ML_Model_tridivided_Output')
